# Transformer：英德机器翻译

这个 Notebook 从零实现 Transformer（Attention Is All You Need，Vaswani et al. 2017）并在 Multi30k 英德翻译数据集上训练。

内容包括：
- 词表构建（word-level tokenization）
- `DataLoader` 构建（含 padding / causal mask）
- Positional Encoding 实现
- Transformer Encoder / Decoder 结构
- Teacher Forcing 训练
- Greedy Decoding 推理
- 翻译效果展示

## 1. 环境准备

```bash
pip install torch datasets
```

In [ ]:
import math
import re
from collections import Counter
from dataclasses import dataclass

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

plt.style.use('seaborn-v0_8')
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
@dataclass
class Config:
    # 训练超参
    batch_size: int = 128
    epochs: int = 10
    lr: float = 3e-4
    # 序列参数
    max_len: int = 50
    min_freq: int = 2      # 词表最低词频
    # 模型结构
    d_model: int = 256
    nhead: int = 8
    num_encoder_layers: int = 3
    num_decoder_layers: int = 3
    d_ff: int = 512
    dropout: float = 0.1


cfg = Config()
cfg

## 2. 加载 Multi30k 数据集

Multi30k 是 Flickr30k 图像描述的英德翻译版本，句子短（平均约 13 词），是翻译模型教学的标准数据集。

In [ ]:
from datasets import load_dataset

# bentrevett/multi30k 是专为教学整理的干净版本
raw = load_dataset('bentrevett/multi30k')
print(raw)
print('\n示例：')
for ex in raw['train'].select(range(3)):
    print(f"  EN: {ex['en']}")
    print(f"  DE: {ex['de']}")
    print()

## 3. 词表构建

In [ ]:
PAD, BOS, EOS, UNK = '<pad>', '<bos>', '<eos>', '<unk>'
SPECIALS = [PAD, BOS, EOS, UNK]


def simple_tokenize(text):
    # 小写 + 按空格和标点拆分
    return re.findall(r"\w+|[^\w\s]", text.lower())


class Vocab:
    def __init__(self, counter, min_freq=1):
        self.token2idx = {tok: i for i, tok in enumerate(SPECIALS)}
        for token, freq in counter.items():
            if freq >= min_freq and token not in self.token2idx:
                self.token2idx[token] = len(self.token2idx)
        self.idx2token = {i: t for t, i in self.token2idx.items()}

    def __len__(self):
        return len(self.token2idx)

    def encode(self, tokens):
        return [self.token2idx.get(t, self.token2idx[UNK]) for t in tokens]

    def decode(self, ids):
        tokens = [self.idx2token.get(i, UNK) for i in ids]
        # 截断到 EOS
        if EOS in tokens:
            tokens = tokens[:tokens.index(EOS)]
        return ' '.join(t for t in tokens if t not in SPECIALS)


# 从训练集统计词频
en_counter, de_counter = Counter(), Counter()
for ex in raw['train']:
    en_counter.update(simple_tokenize(ex['en']))
    de_counter.update(simple_tokenize(ex['de']))

src_vocab = Vocab(en_counter, min_freq=cfg.min_freq)
tgt_vocab = Vocab(de_counter, min_freq=cfg.min_freq)

print(f'EN vocab size: {len(src_vocab)}')
print(f'DE vocab size: {len(tgt_vocab)}')

## 4. Dataset 与 DataLoader

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, split, src_vocab, tgt_vocab, max_len):
        self.data      = raw[split]
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.max_len   = max_len
        self.bos       = tgt_vocab.token2idx[BOS]
        self.eos       = tgt_vocab.token2idx[EOS]
        self.src_bos   = src_vocab.token2idx[BOS]
        self.src_eos   = src_vocab.token2idx[EOS]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ex  = self.data[idx]
        src = [self.src_bos] + self.src_vocab.encode(simple_tokenize(ex['en']))[:self.max_len] + [self.src_eos]
        tgt = [self.bos]     + self.tgt_vocab.encode(simple_tokenize(ex['de']))[:self.max_len] + [self.eos]
        return torch.tensor(src, dtype=torch.long), torch.tensor(tgt, dtype=torch.long)


def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    # pad_sequence 默认在末尾补 0（PAD idx = 0）
    src_pad = pad_sequence(src_batch, batch_first=True, padding_value=0)
    tgt_pad = pad_sequence(tgt_batch, batch_first=True, padding_value=0)
    return src_pad, tgt_pad


train_ds = TranslationDataset('train',      src_vocab, tgt_vocab, cfg.max_len)
val_ds   = TranslationDataset('validation', src_vocab, tgt_vocab, cfg.max_len)
test_ds  = TranslationDataset('test',       src_vocab, tgt_vocab, cfg.max_len)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn)

src_batch, tgt_batch = next(iter(train_loader))
print('src batch:', src_batch.shape)
print('tgt batch:', tgt_batch.shape)

## 5. Transformer 实现

原始 Transformer 采用 Encoder-Decoder 结构：
- **Encoder**：多层 Self-Attention + FFN，对源语言序列建立全局表示
- **Decoder**：多层 Masked Self-Attention + Cross-Attention + FFN，逐 token 生成目标语言
- **Positional Encoding**：正弦余弦编码，让模型感知 token 的绝对位置

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        # 预计算正弦余弦位置编码，形状 (max_len, d_model)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        # 不同维度的频率用对数域计算，数值更稳定
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        # 注册为 buffer，不参与梯度更新
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        # x: (B, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

In [ ]:
class TranslationTransformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, nhead,
                 num_encoder_layers, num_decoder_layers, d_ff, dropout, max_len):
        super().__init__()
        self.src_embed = nn.Embedding(src_vocab_size, d_model, padding_idx=0)
        self.tgt_embed = nn.Embedding(tgt_vocab_size, d_model, padding_idx=0)
        self.pos_enc   = PositionalEncoding(d_model, dropout, max_len)
        # 使用 PyTorch 内置 Transformer（batch_first=True 更直观）
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True,   # Pre-LN，更稳定
        )
        self.head = nn.Linear(d_model, tgt_vocab_size)
        # 权重共享：embedding 和 lm head 复用同一矩阵
        self.head.weight = self.tgt_embed.weight
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def encode(self, src, src_key_padding_mask):
        x = self.pos_enc(self.src_embed(src) * math.sqrt(self.src_embed.embedding_dim))
        return self.transformer.encoder(x, src_key_padding_mask=src_key_padding_mask)

    def decode(self, tgt, memory, tgt_mask, tgt_key_padding_mask, memory_key_padding_mask):
        x = self.pos_enc(self.tgt_embed(tgt) * math.sqrt(self.tgt_embed.embedding_dim))
        return self.transformer.decoder(
            x, memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask,
        )

    def forward(self, src, tgt):
        # padding mask：PAD 位置为 True，让 attention 忽略
        src_pad_mask = (src == 0)
        tgt_pad_mask = (tgt == 0)
        # causal mask：让 decoder 不看到未来 token
        tgt_seq_len = tgt.size(1)
        tgt_causal_mask = nn.Transformer.generate_square_subsequent_mask(tgt_seq_len, device=src.device)

        memory = self.encode(src, src_pad_mask)
        out    = self.decode(tgt, memory, tgt_causal_mask, tgt_pad_mask, src_pad_mask)
        return self.head(out)  # (B, tgt_len, tgt_vocab_size)


model = TranslationTransformer(
    src_vocab_size=len(src_vocab),
    tgt_vocab_size=len(tgt_vocab),
    d_model=cfg.d_model,
    nhead=cfg.nhead,
    num_encoder_layers=cfg.num_encoder_layers,
    num_decoder_layers=cfg.num_decoder_layers,
    d_ff=cfg.d_ff,
    dropout=cfg.dropout,
    max_len=cfg.max_len + 10,
).to(device)
model

## 6. Tensor 尺寸分析

In [ ]:
@torch.no_grad()
def inspect_shapes(model, src_len=10, tgt_len=8, batch=2):
    src = torch.randint(4, len(src_vocab), (batch, src_len))
    tgt = torch.randint(4, len(tgt_vocab), (batch, tgt_len))
    print(f'src input      : {tuple(src.shape)}')
    print(f'tgt input      : {tuple(tgt.shape)}')

    emb = model.src_embed(src) * math.sqrt(cfg.d_model)
    emb = model.pos_enc(emb)
    print(f'src embedded   : {tuple(emb.shape)}')

    memory = model.encode(src, src == 0)
    print(f'encoder memory : {tuple(memory.shape)}')

    causal = nn.Transformer.generate_square_subsequent_mask(tgt_len)
    dec = model.decode(tgt, memory, causal, tgt == 0, src == 0)
    print(f'decoder output : {tuple(dec.shape)}')

    logits = model.head(dec)
    print(f'logits         : {tuple(logits.shape)}')


inspect_shapes(model.cpu())
model = model.to(device)

## 7. 关键机制解读

### Encoder Self-Attention
- 源语言每个 token 都可以关注所有其他 token，建立双向全局上下文。

### Decoder Masked Self-Attention
- 生成时只能看到已生成的 token（causal mask），防止信息泄露。

### Cross-Attention（Encoder-Decoder Attention）
- Decoder 的 Q 来自 Decoder 自身，K/V 来自 Encoder 输出（memory）。
- 这是翻译模型的核心：生成每个目标词时，对齐到最相关的源语言 token。

### Positional Encoding
- Transformer 没有递归结构，位置信息需要显式注入。
- 正弦余弦编码：不同频率的周期函数使模型能感知绝对位置和相对距离。

### Teacher Forcing
- 训练时把真实目标序列右移一位作为 decoder 输入，而不是用模型自己的输出。
- 更稳定，但会导致 exposure bias（推理时只能用自己的输出）。

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print(f'Trainable parameters: {count_parameters(model):,}')

## 8. 训练函数

In [ ]:
# label smoothing 让模型对不确定的位置更鲁棒
criterion = nn.CrossEntropyLoss(ignore_index=0, label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=cfg.lr, betas=(0.9, 0.98), eps=1e-9)


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total      = 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        # teacher forcing：tgt[:, :-1] 作为输入，tgt[:, 1:] 作为目标
        logits = model(src, tgt[:, :-1])
        loss   = criterion(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        # 梯度裁剪，防止梯度爆炸
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * src.size(0)
        total      += src.size(0)
    return total_loss / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total      = 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        logits   = model(src, tgt[:, :-1])
        loss     = criterion(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))
        total_loss += loss.item() * src.size(0)
        total      += src.size(0)
    return total_loss / total

## 9. 训练主循环

In [ ]:
history = {'train_loss': [], 'val_loss': []}

for epoch in range(cfg.epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss   = evaluate(model, val_loader, criterion, device)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    print(
        f'Epoch [{epoch + 1}/{cfg.epochs}]  '
        f'train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  '
        f'train_ppl={math.exp(train_loss):.2f}  val_ppl={math.exp(val_loss):.2f}'
    )

In [ ]:
epochs_range = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, history['train_loss'], label='train loss')
axes[0].plot(epochs_range, history['val_loss'],   label='val loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs_range, [math.exp(l) for l in history['train_loss']], label='train ppl')
axes[1].plot(epochs_range, [math.exp(l) for l in history['val_loss']],   label='val ppl')
axes[1].set_title('Perplexity')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. 翻译推理（Greedy Decoding）

In [ ]:
@torch.no_grad()
def greedy_decode(model, src_sentence, src_vocab, tgt_vocab, max_len, device):
    model.eval()
    src_tokens = [src_vocab.token2idx[BOS]] + \
                 src_vocab.encode(simple_tokenize(src_sentence)) + \
                 [src_vocab.token2idx[EOS]]
    src = torch.tensor(src_tokens, dtype=torch.long).unsqueeze(0).to(device)

    # Encoder 只跑一次
    memory  = model.encode(src, src == 0)
    bos_idx = tgt_vocab.token2idx[BOS]
    eos_idx = tgt_vocab.token2idx[EOS]
    tgt     = torch.tensor([[bos_idx]], dtype=torch.long).to(device)

    for _ in range(max_len):
        tgt_len    = tgt.size(1)
        causal     = nn.Transformer.generate_square_subsequent_mask(tgt_len, device=device)
        dec_out    = model.decode(tgt, memory, causal, tgt == 0, src == 0)
        next_token = model.head(dec_out[:, -1]).argmax(dim=-1, keepdim=True)
        tgt        = torch.cat([tgt, next_token], dim=1)
        if next_token.item() == eos_idx:
            break

    return tgt_vocab.decode(tgt[0].tolist())


test_sentences = [
    'A dog is running in the park .',
    'Two children are playing with a ball .',
    'A woman is reading a book on a bench .',
]

print('EN → DE 翻译结果：\n')
for sent in test_sentences:
    trans = greedy_decode(model, sent, src_vocab, tgt_vocab, cfg.max_len, device)
    print(f'  EN: {sent}')
    print(f'  DE: {trans}')
    print()